<h1 align=center><font size = 5>Assignment: SQL Notebook for Peer Assignment</font></h1>

## Introduction
Using this Python notebook you will:

1.  Understand the Spacex DataSet
2.  Load the dataset  into the corresponding table in a Db2 database
3.  Execute SQL queries to answer assignment questions 

## Overview of the DataSet

SpaceX has gained worldwide attention for a series of historic milestones. 

It is the only private company ever to return a spacecraft from low-earth orbit, which it first accomplished in December 2010.
SpaceX advertises Falcon 9 rocket launches on its website with a cost of 62 million dollars wheras other providers cost upward of 165 million dollars each, much of the savings is because Space X can reuse the first stage. 


Therefore if we can determine if the first stage will land, we can determine the cost of a launch. 

This information can be used if an alternate company wants to bid against SpaceX for a rocket launch.

This dataset includes a record for each payload carried during a SpaceX mission into outer space.


### Download the datasets

This assignment requires you to load the spacex dataset.

In many cases the dataset to be analyzed is available as a .CSV (comma separated values) file, perhaps on the internet. Click on the link below to download and save the dataset (.CSV file):

 <a href="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_2/data/Spacex.csv" target="_blank">Spacex DataSet</a>



In [1]:
# omitida:
# !pip install sqlalchemy==1.3.9

### Connect to the database

Let us first load the SQL extension and establish a connection with the database


In [2]:
# omitidas:
# !pip install ipython-sql
# !pip install ipython-sql prettytable

# %load_ext sql

In [3]:
#--- HEADER: LIBRARY IMPORTS FOR SQL DATABASE
#--- Importing libraries for CSV handling, SQLite database operations,
#--- and pretty-printing query results.

import csv, sqlite3          # csv for reading CSV files, sqlite3 for database operations
import prettytable           # prettytable for displaying query results in a clean tabular format

In [4]:
#--- HEADER: CONFIGURE PRETTYTABLE AND CONNECT TO DATABASE
#--- Setting the default table style for prettytable and establishing
#--- a connection to the SQLite database.

# Set the default table style for prettytable (e.g., 'DEFAULT', 'MSWORD_FRIENDLY', etc.)
prettytable.DEFAULT = 'DEFAULT'

# Connect to the SQLite database file (creates it if it doesn't exist)
con = sqlite3.connect("C10M2_my_data1.db")

# Create a cursor object to execute SQL commands
cur = con.cursor()

In [5]:
# NOTE mìo, ver:
# To view the SQLite database my_data1.db in DBeaver on your Linux machine, you need to create a new SQLite connection and point it to your database file.
# DONE!
# /home/mapa8/Documents/CIDSPC/C10M2_my_data1.db

In [6]:
# omitida:
# !pip install -q pandas
# %sql sqlite:///my_data1.db

In [7]:
#--- HEADER: LOAD SPACEX DATASET FROM CSV
#--- Loading the SpaceX dataset from a local CSV file into a pandas DataFrame
#--- for analysis and database insertion.

import pandas as pd

df = pd.read_csv("/home/mapa8/Documents/CIDSPC/MFaD/Spacex.csv")

In [8]:
#--- HEADER: VERIFY DATAFRAME
#--- Check the shape and first few rows to confirm successful loading.

print(f"DataFrame shape: {df.shape}")
df.head()

DataFrame shape: (101, 10)


,Date,Time (UTC),Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing_Outcome
0,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
1,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of...",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
3,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
4,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt


In [9]:
#--- HEADER: INSERT DATAFRAME INTO SQLITE TABLE
#--- Writing the pandas DataFrame to a SQLite table named 'SPACEXTBL'.
#--- The table will be replaced if it already exists.

# df.to_sql() writes the entire DataFrame to a SQL table.
# "SPACEXTBL" is the name of the target table in the database.
# con is the SQLite database connection object (established earlier).
# if_exists='replace' drops the table if it already exists and recreates it.
# index=False prevents pandas from adding an extra 'index' column to the table.
# method="multi" inserts rows in batches for faster performance with large datasets.
df.to_sql("SPACEXTBL", con, if_exists='replace', index=False, method="multi")

101

**Note:This below code is added to remove blank rows from table**

In [10]:
# Since using VS Code, here's the equivalent Python code:

#--- HEADER: DROP TABLE IF EXISTS (USING SQLITE3)
#--- Removing the table if it exists to avoid conflicts.

import sqlite3

# Connect to the database
con = sqlite3.connect("C10M2_my_data1.db")
cur = con.cursor()

# Drop table if it exists
cur.execute("DROP TABLE IF EXISTS SPACEXTABLE;")
con.commit()

print("✅ Table SPACEXTABLE dropped successfully (if it existed).")

# Close connection
con.close()

✅ Table SPACEXTABLE dropped successfully (if it existed).


In [11]:
#--- HEADER: CREATE TABLE WITH NON-NULL DATES (SQLITE3 VERSION)
#--- Creating a new table with only rows that have valid dates.

import sqlite3
import pandas as pd

# Connect to the database
con = sqlite3.connect("C10M2_my_data1.db")

# Create new table with non-null dates
query = """
CREATE TABLE SPACEXTABLE AS 
SELECT * FROM SPACEXTBL 
WHERE Date IS NOT NULL;
"""
cur = con.cursor()
cur.execute(query)
con.commit()

print("✅ Table SPACEXTABLE created successfully with non-null dates.")

# Verify the new table
result = pd.read_sql_query("SELECT COUNT(*) FROM SPACEXTABLE;", con)
print(f"Rows in SPACEXTABLE: {result.iloc[0, 0]}")

# Close connection
con.close()

✅ Table SPACEXTABLE created successfully with non-null dates.
Rows in SPACEXTABLE: 101


## Tasks

Now write and execute SQL queries to solve the assignment tasks.

**Note: If the column names are in mixed case enclose it in double quotes
   For Example "Landing_Outcome"**

### Task 1




##### Display the names of the unique launch sites  in the space mission


In [12]:
#--- HEADER: TASK 1 - DISPLAY UNIQUE LAUNCH SITES (SQLITE3 VERSION)
#--- Querying the database to retrieve all unique launch site names.

import sqlite3
import pandas as pd

# Connect to the database
con = sqlite3.connect("C10M2_my_data1.db")

# SELECT DISTINCT "LaunchSite" returns only unique values from the LaunchSite column.
# Double quotes handle the mixed-case column name (required by SQLite).
# FROM SPACEXTABLE specifies the table to query.
# pd.read_sql_query() executes the query and returns results as a DataFrame.
query = 'SELECT DISTINCT "Launch_Site" FROM SPACEXTABLE;'
result = pd.read_sql_query(query, con)

print("Unique Launch Sites:")
print(result)

# Close the database connection
con.close()

Unique Launch Sites:
    Launch_Site
0   CCAFS LC-40
1   VAFB SLC-4E
2    KSC LC-39A
3  CCAFS SLC-40



### Task 2


#####  Display 5 records where launch sites begin with the string 'CCA' 


In [13]:
#--- HEADER: TASK 2 - DISPLAY 5 RECORDS FROM LAUNCH SITES STARTING WITH 'CCA'
#--- Querying the database for 5 records where launch site names begin with 'CCA'.

# Connect to the database (using the existing connection or creating a new one)
con = sqlite3.connect("C10M2_my_data1.db")

# SELECT * retrieves all columns from the table.
# WHERE "Launch_Site" LIKE 'CCA%' filters for launch sites starting with 'CCA'.
# The % wildcard matches any characters after 'CCA'.
# LIMIT 5 restricts the output to only 5 records.
query = 'SELECT * FROM SPACEXTABLE WHERE "Launch_Site" LIKE "CCA%" LIMIT 5;'
result = pd.read_sql_query(query, con)

print("5 Records from launch sites starting with 'CCA':")
print(result)

# Close the database connection
con.close()

5 Records from launch sites starting with 'CCA':
         Date Time (UTC) Booster_Version  Launch_Site  \
0  2010-06-04   18:45:00  F9 v1.0  B0003  CCAFS LC-40   
1  2010-12-08   15:43:00  F9 v1.0  B0004  CCAFS LC-40   
2  2012-05-22    7:44:00  F9 v1.0  B0005  CCAFS LC-40   
3  2012-10-08    0:35:00  F9 v1.0  B0006  CCAFS LC-40   
4  2013-03-01   15:10:00  F9 v1.0  B0007  CCAFS LC-40   

                                             Payload  PAYLOAD_MASS__KG_  \
0               Dragon Spacecraft Qualification Unit                  0   
1  Dragon demo flight C1, two CubeSats, barrel of...                  0   
2                              Dragon demo flight C2                525   
3                                       SpaceX CRS-1                500   
4                                       SpaceX CRS-2                677   

       Orbit         Customer Mission_Outcome      Landing_Outcome  
0        LEO           SpaceX         Success  Failure (parachute)  
1  LEO (ISS)  NASA 

### Task 3




##### Display the total payload mass carried by boosters launched by NASA (CRS)


In [14]:
#--- HEADER: TASK 3 - DISPLAY TOTAL PAYLOAD MASS FOR NASA (CRS) MISSIONS
#--- Querying the total payload mass carried by boosters launched by NASA (CRS missions).

# Connect to the database
con = sqlite3.connect("C10M2_my_data1.db")

# SUM("PAYLOAD_MASS__KG_") calculates the total payload mass.
# WHERE "Customer" LIKE '%NASA%' filters for NASA-related missions.
# The % wildcard matches any characters before and after 'NASA'.
# 'CRS' (Commercial Resupply Services) missions are NASA's cargo missions.
query = 'SELECT SUM("PAYLOAD_MASS__KG_") AS "Total_Payload_Mass_KG" FROM SPACEXTABLE WHERE "Customer" LIKE "%NASA%" OR "Customer" LIKE "%CRS%";'
result = pd.read_sql_query(query, con)

print("Total Payload Mass for NASA (CRS) Missions:")
print(result)

# Close the database connection
con.close()

Total Payload Mass for NASA (CRS) Missions:
   Total_Payload_Mass_KG
0                 107010


### Task 4




##### Display average payload mass carried by booster version F9 v1.1


In [15]:
#--- HEADER: TASK 4 - AVERAGE PAYLOAD MASS FOR F9 v1.1 (WITH LIKE)
#--- Using LIKE in case the version string has variations (e.g., 'F9 v1.1', 'F9 v1.1 (FT)').

con = sqlite3.connect("C10M2_my_data1.db")

# LIKE '%F9 v1.1%' catches any variations containing the core version.
query = 'SELECT AVG("PAYLOAD_MASS__KG_") AS "Average_Payload_Mass_KG" FROM SPACEXTABLE WHERE "Booster_Version" LIKE "%F9 v1.1%";'
result = pd.read_sql_query(query, con)

print("Average Payload Mass for Booster Version F9 v1.1 (with variations):")
print(result)

con.close()

Average Payload Mass for Booster Version F9 v1.1 (with variations):
   Average_Payload_Mass_KG
0              2534.666667


### Task 5

##### List the date when the first succesful landing outcome in ground pad was acheived.


_Hint:Use min function_ 


In [16]:
#--- HEADER: TASK 5 - FIRST SUCCESSFUL GROUND PAD LANDING (WITH LIKE)
#--- Using LIKE in case the landing outcome string has variations.

con = sqlite3.connect("C10M2_my_data1.db")

# LIKE '%Success%ground pad%' catches variations containing success and ground pad.
query = 'SELECT MIN("Date") AS "First_Successful_Ground_Pad_Landing_Date" FROM SPACEXTABLE WHERE "Landing_Outcome" LIKE "%Success%";' # NOTE modificado a solo "Success"
result = pd.read_sql_query(query, con)

print("Date of the first successful landing on a ground pad:")
print(result)

con.close()

Date of the first successful landing on a ground pad:
  First_Successful_Ground_Pad_Landing_Date
0                               2015-12-22


### Task 6

##### List the names of the boosters which have success in drone ship and have payload mass greater than 4000 but less than 6000


In [17]:
#--- HEADER: TASK 6 - DETAILED VIEW OF DRONE SHIP SUCCESS WITH PAYLOAD MASS 4000-6000 KG
#--- Showing all records (not just distinct boosters) for more detail.

con = sqlite3.connect("C10M2_my_data1.db")

# SELECT * shows all columns for each matching record.
query = '''
SELECT * 
FROM SPACEXTABLE 
WHERE "Landing_Outcome" LIKE '%drone ship%' 
  AND "PAYLOAD_MASS__KG_" BETWEEN 4000 AND 6000;
'''
result = pd.read_sql_query(query, con)

print("All records with drone ship landings and payload mass between 4000-6000 kg:")
print(result)

con.close()

All records with drone ship landings and payload mass between 4000-6000 kg:
         Date Time (UTC) Booster_Version  Launch_Site                Payload  \
0  2016-03-04   23:35:00     F9 FT B1020  CCAFS LC-40                  SES-9   
1  2016-05-06    5:21:00     F9 FT B1022  CCAFS LC-40               JCSAT-14   
2  2016-08-14    5:26:00     F9 FT B1026  CCAFS LC-40               JCSAT-16   
3  2017-03-30   22:27:00  F9 FT  B1021.2   KSC LC-39A                 SES-10   
4  2017-10-11   22:53:00  F9 FT  B1031.2   KSC LC-39A  SES-11 / EchoStar 105   

   PAYLOAD_MASS__KG_ Orbit                Customer Mission_Outcome  \
0               5271   GTO                     SES         Success   
1               4696   GTO  SKY Perfect JSAT Group         Success   
2               4600   GTO  SKY Perfect JSAT Group         Success   
3               5300   GTO                     SES         Success   
4               5200   GTO            SES EchoStar         Success   

        Landing_Outcom

### Task 7




##### List the total number of successful and failure mission outcomes


In [18]:
#--- HEADER: TASK 7 - LIST TOTAL NUMBER OF SUCCESSFUL AND FAILURE MISSION OUTCOMES
#--- Querying the count of mission outcomes grouped by success and failure.

# Connect to the database
con = sqlite3.connect("C10M2_my_data1.db")

# "Mission_Outcome" is the column containing mission results.
# COUNT(*) counts the number of rows in each group.
# GROUP BY "Mission_Outcome" groups the results by each unique outcome.
query = '''
SELECT "Mission_Outcome", COUNT(*) AS "Total_Count" 
FROM SPACEXTABLE 
GROUP BY "Mission_Outcome";
'''
result = pd.read_sql_query(query, con)

print("Total Number of Mission Outcomes:")
print(result)

# Close the database connection
con.close()

Total Number of Mission Outcomes:
                    Mission_Outcome  Total_Count
0               Failure (in flight)            1
1                           Success           98
2                          Success             1
3  Success (payload status unclear)            1


### Task 8



##### List all the booster_versions that have carried the maximum payload mass, using a subquery with a suitable aggregate function.


In [19]:
#--- HEADER: TASK 8 - LIST BOOSTER VERSIONS WITH MAXIMUM PAYLOAD MASS
#--- Using a subquery to find the booster version(s) that have carried
#--- the maximum payload mass.

# Connect to the database
con = sqlite3.connect("C10M2_my_data1.db")

# SELECT "Booster_Version", "PAYLOAD_MASS__KG_" selects the booster and its payload mass.
# WHERE "PAYLOAD_MASS__KG_" = (SELECT MAX("PAYLOAD_MASS__KG_") FROM SPACEXTABLE)
# The subquery finds the highest payload mass in the entire table.
# This ensures we get ALL boosters that have carried that maximum mass.
query = '''
SELECT "Booster_Version", "PAYLOAD_MASS__KG_" 
FROM SPACEXTABLE 
WHERE "PAYLOAD_MASS__KG_" = (SELECT MAX("PAYLOAD_MASS__KG_") FROM SPACEXTABLE);
'''
result = pd.read_sql_query(query, con)

print("Booster version(s) that carried the maximum payload mass:")
print(result)

# Close the database connection
con.close()

Booster version(s) that carried the maximum payload mass:
   Booster_Version  PAYLOAD_MASS__KG_
0    F9 B5 B1048.4              15600
1    F9 B5 B1049.4              15600
2    F9 B5 B1051.3              15600
3    F9 B5 B1056.4              15600
4    F9 B5 B1048.5              15600
5    F9 B5 B1051.4              15600
6    F9 B5 B1049.5              15600
7   F9 B5 B1060.2               15600
8   F9 B5 B1058.3               15600
9    F9 B5 B1051.6              15600
10   F9 B5 B1060.3              15600
11  F9 B5 B1049.7               15600


### Task 9


##### List the records which will display the month names, failure landing_outcomes in drone ship ,booster versions, launch_site for the months in year 2015.

**Note: SQLLite does not support monthnames. So you need to use  substr(Date, 6,2) as month to get the months and substr(Date,0,5)='2015' for year.**


In [20]:
#--- HEADER: TASK 9 - LIST FAILURE LANDING OUTCOMES ON DRONE SHIP IN 2015
#--- Querying records with failure landing outcomes on drone ships,
#--- showing booster versions, launch sites, and month names for the year 2015.
#--- Note: SQLite does not have a MONTHNAME() function, so we use substr().

# Connect to the database
con = sqlite3.connect("C10M2_my_data1.db")

# substr("Date", 6, 2) extracts the month as a two-digit string (e.g., '01', '06').
# CASE statement converts month numbers to month names for readability.
# substr("Date", 1, 4) = '2015' filters for the year 2015.
# "Landing_Outcome" LIKE '%failure%' AND '%drone ship%' filters for failure on drone ship.
query = '''
SELECT 
    CASE substr("Date", 6, 2)
        WHEN '01' THEN 'January'
        WHEN '02' THEN 'February'
        WHEN '03' THEN 'March'
        WHEN '04' THEN 'April'
        WHEN '05' THEN 'May'
        WHEN '06' THEN 'June'
        WHEN '07' THEN 'July'
        WHEN '08' THEN 'August'
        WHEN '09' THEN 'September'
        WHEN '10' THEN 'October'
        WHEN '11' THEN 'November'
        WHEN '12' THEN 'December'
    END AS "Month",
    "Landing_Outcome",
    "Booster_Version",
    "Launch_Site"
FROM SPACEXTABLE 
WHERE substr("Date", 1, 4) = '2015'
  AND "Landing_Outcome" LIKE '%failure%' 
  AND "Landing_Outcome" LIKE '%drone ship%';
'''
result = pd.read_sql_query(query, con)

print("Failure landing outcomes on drone ship in 2015:")
print(result)

# Close the database connection
con.close()

Failure landing outcomes on drone ship in 2015:
     Month       Landing_Outcome Booster_Version  Launch_Site
0  January  Failure (drone ship)   F9 v1.1 B1012  CCAFS LC-40
1    April  Failure (drone ship)   F9 v1.1 B1015  CCAFS LC-40


### Task 10




##### Rank the count of landing outcomes (such as Failure (drone ship) or Success (ground pad)) between the date 2010-06-04 and 2017-03-20, in descending order.


In [21]:
#--- HEADER: TASK 10 - RANK COUNT OF LANDING OUTCOMES BY DATE RANGE
#--- Counting and ranking landing outcomes between 2010-06-04 and 2017-03-20.
#--- Results are ordered in descending order (highest count first).

# Connect to the database
con = sqlite3.connect("C10M2_my_data1.db")

# SELECT "Landing_Outcome", COUNT(*) AS "Count" selects the outcome and its count.
# WHERE "Date" BETWEEN '2010-06-04' AND '2017-03-20' filters for the date range.
# GROUP BY "Landing_Outcome" groups the results by each unique landing outcome.
# ORDER BY "Count" DESC sorts from highest to lowest count.
query = '''
SELECT 
    "Landing_Outcome", 
    COUNT(*) AS "Count"
FROM SPACEXTABLE 
WHERE "Date" BETWEEN '2010-06-04' AND '2017-03-20'
GROUP BY "Landing_Outcome"
ORDER BY "Count" DESC;
'''
result = pd.read_sql_query(query, con)

print("Ranked count of landing outcomes (2010-06-04 to 2017-03-20):")
print(result)

# Close the database connection
con.close()

Ranked count of landing outcomes (2010-06-04 to 2017-03-20):
          Landing_Outcome  Count
0              No attempt     10
1    Success (drone ship)      5
2    Failure (drone ship)      5
3    Success (ground pad)      3
4      Controlled (ocean)      3
5    Uncontrolled (ocean)      2
6     Failure (parachute)      2
7  Precluded (drone ship)      1
